# Train the diffusion model on a Colab GPU

Run these cells **in order**. Before you start: **Runtime → Change runtime type → T4 GPU**.

This notebook doesn't contain the model code. It downloads *your* code from GitHub and runs it,
so write and push your steps first (`git push`), then run `git pull` here to get the latest version.

## 1. Check that the GPU is on
You should see a Tesla T4. If it says "command not found", the GPU is off.

In [ ]:
!nvidia-smi

## 2. Connect Google Drive
Checkpoints are saved here, so a Colab disconnect doesn't lose hours of training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Get your code from GitHub
Run this cell again (the `git pull` part) whenever you push new code.

In [ ]:
import os
if not os.path.exists('/content/learning-ai-at-its-core'):
    !git clone https://github.com/Tehman700/learning-ai-at-its-core.git /content/learning-ai-at-its-core
%cd "/content/learning-ai-at-its-core/Image Generation Diffusion Model"
!git checkout -- . && git pull   # throws away edits made inside Colab (the sed lines below); your real code comes from GitHub
!pip install -q gradio tqdm

## 4. Check the dataset
`config.py` decides `DATASET`. It is now **`"dogs"`** by default (the MNIST practice round is done).
To train MNIST again, uncomment the line below. After changing it: **Runtime → Restart session**, then run from cell 2.

In [ ]:
# Uncomment to train MNIST instead:
# !sed -i 's/^DATASET = "dogs"/DATASET = "mnist"/' config.py
!grep '^DATASET' config.py

## 5. Download Stanford Dogs
About 740 MB, from the Hugging Face copy (the original Stanford server is often offline).
It goes on Colab's own fast disk (not Drive), so you need to re-run this after every new Colab session.

In [ ]:
!mkdir -p data/stanford_dogs
%cd data/stanford_dogs
!wget -q https://huggingface.co/datasets/Alanox/stanford-dogs/resolve/main/metadata.csv
!wget -q https://huggingface.co/datasets/Alanox/stanford-dogs/resolve/main/images.tar.gz
!tar -xzf images.tar.gz && rm images.tar.gz && echo "images: $(ls images | wc -l)"
%cd ../..

## 6. Save checkpoints to Google Drive, then train
Set the path *before* importing the training code.

In [ ]:
from pathlib import Path
import config
config.CHECKPOINT_DIR = Path('/content/drive/MyDrive/diffusion_checkpoints')
config.SAMPLES_DIR = Path('/content/drive/MyDrive/diffusion_samples')

from step4_train import train
_ = train()   # "_ =" stops Colab from printing the whole model when training ends

## 7. Look at the latest preview grid

In [ ]:
from IPython.display import Image, display
latest = max(config.SAMPLES_DIR.glob('*_epoch_*.png'), key=lambda p: p.stat().st_mtime)   # the newest preview
print(latest.name)
display(Image(str(latest), width=400))

## 8. (Step 7) Run the app with a public link

In [ ]:
!sed -i 's/app.launch()/app.launch(share=True)/' step7_app.py
!python step7_app.py